# 02 - Putting detections on the ground

A detection is a box in a frame. A survey needs it on the map. Between the two sit a camera pose, a terrain model and one piece of geometry: cast a ray through the pixel and see where it meets the ground. This notebook does that for real annotations of a public flight, then - because every public flight looks straight down - builds a synthetic oblique scene to show what the pointing maths has to get right at 45 and 80 degrees, and what the old code got wrong.

Everything speaks numpy: `(N, 2)` pixels in, `(N, 3)` DEM-local metres out, NaN where a ray missed. Run `00_setup` first if you have not; `01_frames_and_poses` explains where the poses come from.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
from _setup import ensure_environment, get_flight, get_dem, find
ensure_environment()

import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

## Part 1 - Flight 146: wild boar annotations onto a real DEM

### The terrain

The dataset tooling (`dem_from_poses.py`) cuts a DEM around the flight and writes it as a triangle mesh (`.glb`) plus a small metadata JSON with the raster origin. `bambi.io.dem` reads both; the mesh is DEM-local metres with the origin at the south-west corner and at the lowest elevation. (`get_dem` uses the same tool - or, to keep CI off a 10 GB tile download, meshes a 0.4 MB GeoTIFF clip shipped with the repo through `bambi.io.dem.geotiff_to_dem`. Same layout either way.)

In [ ]:
from bambi.io.dem import read_dem_mesh

flight = get_flight("146", version="base")
dem = read_dem_mesh(get_dem("146"))
v = dem.vertices
print(f"mesh: {len(v):,} vertices, {len(dem.faces):,} triangles")
print(f"extent: {v[:,0].max():.0f} m east x {v[:,1].max():.0f} m north, relief {v[:,2].max():.1f} m")
print(f"origin: {dem.origin.latitude:.5f} N {dem.origin.longitude:.5f} E, {dem.origin.altitude:.1f} m, EPSG:{dem.origin.epsg}")

### Poses in the DEM's frame

The public poses are geographic; `to_local_poses` puts them into the DEM's frame when told which origin to use - **the DEM's**, not the one written in the poses file (that one is a placeholder). The correction file next to the poses carries the small per-flight offsets the annotators calibrated (a heading nudge in radians, a height offset in metres, with one frame-range override); `bambi.io.corrections` expands it to per-frame arrays.

In [ ]:
from bambi.io.poses import read_poses, to_local_poses
from bambi.io.corrections import read_corrections, corrections_for_frames

pf = read_poses(find(flight, "*_matched_poses.json")[0])
poses = to_local_poses(pf, epsg=dem.origin.epsg, origin=dem.origin)
corr = read_corrections(flight / "146_correction.json")
t_corr, r_corr = corrections_for_frames(corr, len(poses))
print(f"{len(poses)} poses; drone {poses.positions[:,2].min():.0f}..{poses.positions[:,2].max():.0f} m above the DEM origin")
print("correction:", corr.translation, "m,", np.degrees(corr.rotation).round(3), "deg,", len(corr.ranges), "frame-range override(s)")

### The annotations

`146_gt.txt` is MOT format: `frame, track, left, top, width, height, ...`. The frame number indexes the poses directly, and pixel coordinates are in the thermal half of the processed video (1024 x 1024). The processed video is thermal|RGB side by side, so a frame's thermal image is its left half.

In [ ]:
import cv2

W = H = 1024
gt = np.array([ln.split(",")[:6] for ln in open(flight / "146_gt.txt")], dtype=float)
frames = np.unique(gt[:, 0]).astype(int)
print(f"{len(gt)} annotations on {len(frames)} frames ({frames.min()}..{frames.max()}); species: wild boar")

def thermal_frame(idx):
    cap = cv2.VideoCapture(str(flight / "146_matched_processed.mp4"))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, img = cap.read(); cap.release()
    return img[:, :W, ::-1] if ok else None

demo = int(frames[len(frames) // 2])
rows = gt[gt[:, 0] == demo]
boxes = np.column_stack([rows[:, 2], rows[:, 3], rows[:, 2] + rows[:, 4], rows[:, 3] + rows[:, 5]])   # x1 y1 x2 y2

img = thermal_frame(demo)
fig, ax = plt.subplots(figsize=(6, 6)); ax.imshow(img); ax.axis("off")
for x1, y1, x2, y2 in boxes:
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="yellow", lw=1.5))
ax.set_title(f"frame {demo}: {len(boxes)} annotated boar"); plt.show()

### One camera per frame, then rays

`bambi.geo.camera.cameras_from_poses` turns pose rows into alfspy cameras. It owns the two conventions that have bitten this project: the pose rotation (`[tilt, roll, heading]`, heading applied about world *up* after the tilt) and the ray convention of whichever alfspy build is installed (probed once, so both the ModernGL and the PyTorch backend give the same world ray). Corrections are applied exactly once - the same way the renderer applies them.

`bambi.geo.georef` then casts rays: `footprint` for the whole frame, `boxes_to_world` for the four corners of each box.

In [ ]:
from bambi.geo.camera import cameras_from_poses, ray_convention
from bambi.geo import georef

print("installed alfspy ray convention:", ray_convention())
fovy = 50.0          # the dataset's thermal camera; the poses file has no fovy of its own
mesh = dem.mesh

camera = cameras_from_poses(poses, fovy, aspect_ratio=W / H,
                            translation_corrections=t_corr, rotation_corrections=r_corr, indices=[demo])[0]
fp = georef.footprint(camera, W, H, mesh, samples_per_edge=8)          # (32, 3) ground polygon
corners = georef.boxes_to_world(boxes, camera, W, H, mesh)              # (n, 4, 3)
centres = corners.mean(axis=1)
ground = poses.positions[demo, 2] - fp[:, 2].mean()
print(f"drone {ground:.0f} m above the ground it sees; footprint {np.ptp(fp[:,0]):.0f} x {np.ptp(fp[:,1]):.0f} m")
print("box centres (DEM-local east, north, up):"); print(centres.round(2))

In [ ]:
def show_map(ax, extent_pad=15):
    z = dem.mesh.vertices[:, 2]
    tri = ax.tripcolor(v[:, 0], v[:, 1], dem.faces, z, cmap="terrain", shading="gouraud", alpha=0.9)
    ax.set_aspect("equal"); ax.set_xlabel("east (m)"); ax.set_ylabel("north (m)")
    return tri

fig, ax = plt.subplots(figsize=(7, 6.5))
tri = show_map(ax)
ax.add_patch(MplPolygon(fp[:, :2], closed=True, fill=False, color="white", lw=1.5, label="frame footprint"))
ax.plot(*poses.positions[demo, :2], "k^", ms=9, label="drone")
for cn in corners:
    ax.add_patch(MplPolygon(cn[:, :2], closed=True, fill=True, color="yellow", alpha=0.8))
ax.plot(centres[:, 0], centres[:, 1], "r.", ms=4, label="boar")
pad = 40; cx, cy = poses.positions[demo, :2]
ax.set_xlim(cx - pad, cx + pad); ax.set_ylim(cy - pad, cy + pad)
ax.legend(loc="upper right"); plt.colorbar(tri, label="height above DEM origin (m)")
ax.set_title(f"frame {demo} on the DEM"); plt.show()

### All 57 frames

The same call, batched over every annotated frame. The animals barely move over the annotated window - the boxes from consecutive frames pile up on the same few metres of ground, which is exactly what makes a track geo-referenceable rather than 240 separate points.

In [ ]:
cams = cameras_from_poses(poses, fovy, aspect_ratio=W / H,
                          translation_corrections=t_corr, rotation_corrections=r_corr, indices=frames)
all_centres, all_frames = [], []
for fi, cam in zip(frames, cams):
    r = gt[gt[:, 0] == fi]
    b = np.column_stack([r[:, 2], r[:, 3], r[:, 2] + r[:, 4], r[:, 3] + r[:, 5]])
    all_centres.append(georef.boxes_to_world(b, cam, W, H, mesh).mean(axis=1))
    all_frames.append(np.full(len(b), fi))
all_centres = np.vstack(all_centres); all_frames = np.concatenate(all_frames)
missed = np.isnan(all_centres).any(axis=1)
print(f"{len(all_centres)} boxes -> ground; {missed.sum()} missed the DEM")
print(f"they occupy {np.ptp(all_centres[:,0]):.1f} m east x {np.ptp(all_centres[:,1]):.1f} m north")

fig, ax = plt.subplots(figsize=(7, 6.5))
tri = show_map(ax)
sc = ax.scatter(all_centres[:, 0], all_centres[:, 1], c=all_frames, s=10, cmap="plasma")
ax.plot(poses.positions[frames, 0], poses.positions[frames, 1], "k-", lw=1, label="drone path")
ax.set_xlim(cx - pad, cx + pad); ax.set_ylim(cy - pad, cy + pad); ax.legend()
plt.colorbar(sc, label="frame"); ax.set_title("every annotation, on the ground"); plt.show()

### Out to the world

DEM-local metres are for computing; a GIS wants coordinates. `local_to_geographic` inverts the origin, so a GeoJSON is a few lines. (Writing files is deliberately not the engine's business - it hands you arrays.)

In [ ]:
from bambi.geo.poses import local_to_geographic

lla = local_to_geographic(all_centres[~missed], dem.origin)          # (n, 3) lat, lon, alt
features = [{"type": "Feature", "geometry": {"type": "Point", "coordinates": [float(lon), float(lat), float(alt)]},
             "properties": {"frame": int(f)}} for (lat, lon, alt), f in zip(lla, all_frames[~missed])]
out = flight / "146_boar_georeferenced.geojson"
out.write_text(json.dumps({"type": "FeatureCollection", "features": features}))
print(f"wrote {len(features)} points to {out.name}; first: {lla[0].round(6)}")

## Part 2 - Oblique views, on a scene where the truth is exact

Every public BAMBI flight is nadir, so nothing above could tell a right heading convention from a wrong one: at nadir the heading only spins the image. The one oblique flight this project has seen is not ours to publish. So `bambi.testing.synthetic` builds a scene instead: rolling terrain, markers at known coordinates *on* that terrain, and cameras looking at one aim point from 0, 45 and 80 degrees of tilt on three headings.

The test is the round trip: project a marker into a camera (`world_to_pixel`), cast that pixel back onto the mesh (`pixels_to_world`), and demand the marker back.

In [ ]:
from bambi.testing.synthetic import make_scene, visible_markers
from bambi.geo.camera import world_to_pixel

scene = make_scene()                    # tilts (0, 45, 80) x headings (0, 120, 250), 12 markers
sv = scene.vertices
fig, ax = plt.subplots(figsize=(7, 6))
tri = ax.tripcolor(sv[:, 0], sv[:, 1], scene.faces, sv[:, 2], cmap="terrain", shading="gouraud")
ax.plot(scene.markers[:, 0], scene.markers[:, 1], "r.", ms=6, label="markers")
P = scene.poses.positions
ax.scatter(P[:, 0], P[:, 1], c=scene.poses.rotations[:, 0], cmap="cool", s=50, marker="^", label="cameras (colour = tilt)")
for p, r in zip(P, scene.poses.rotations):
    ax.annotate(f"{r[0]:.0f}/{r[2]:.0f}", p[:2], fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_aspect("equal"); ax.legend(loc="lower left"); plt.colorbar(tri, label="height (m)")
ax.set_title("synthetic scene: tilt/heading at each camera, all aimed at the centre"); plt.show()

In [ ]:
w, h = scene.frame_size
smesh = scene.mesh
cams = cameras_from_poses(scene.poses, scene.fovy)
print(f"{'tilt':>5} {'heading':>8} {'in view':>8} {'visible':>8} {'max error (m)':>14}")
for i, cam in enumerate(cams):
    visible, px = visible_markers(scene, i, smesh)                       # occlusion-aware
    in_view = np.isfinite(px).all(axis=1).sum()
    back = georef.pixels_to_world(px[visible], cam, w, h, smesh)
    err = np.linalg.norm(back - scene.markers[visible], axis=1)
    tilt, heading = scene.poses.rotations[i, [0, 2]]
    print(f"{tilt:5.0f} {heading:8.0f} {in_view:8d} {visible.sum():8d} {err.max():14.6f}")

Millimetres everywhere (the mesh vertices are float32). Where fewer markers are *visible* than *in view*, a hill in front of the marker was hit first - the ray-caster is right to stop there, and `visible_markers` says so instead of counting it as an error.

### And the way it used to be done

Before alfspy 2.1 every caller composed the pose rotation as `quaternion_from_eulers([tilt, roll, heading], 'zyx')`, which applies the heading about the camera's own optical axis instead of world up. Same code, old rotation: cast the centre pixel of each camera and measure how far from the aim point it lands.

In [ ]:
from pyrr import Vector3
from alfspy.core.rendering import Camera
from alfspy.core.util.pyrrs import quaternion_from_eulers

rows = []
for pos, rot in zip(scene.poses.positions, scene.poses.rotations):
    q = quaternion_from_eulers([np.deg2rad(a) for a in rot], "zyx")
    if ray_convention() == "legacy":
        q = q.conjugate
    old = Camera(fovy=scene.fovy, aspect_ratio=1.0, position=Vector3(pos), rotation=q)
    hit = georef.pixels_to_world([w / 2, h / 2], old, w, h, smesh)[0]
    miss = np.linalg.norm(hit - scene.aim) if np.isfinite(hit).all() else np.inf
    rows.append((rot[0], rot[2], miss))
rows = np.array(rows)
print(f"{'tilt':>5} {'heading':>8} {'centre ray misses aim by (m)':>30}")
for t, hd, m in rows:
    print(f"{t:5.0f} {hd:8.0f} {('off the mesh' if np.isinf(m) else f'{m:.2f}'):>30}")

At nadir the old spelling is indistinguishable from the right one - which is how it survived years of top-down surveys. Tilt the gimbal and it is tens of metres off, or not on the terrain at all (on heading 0 it even tilts the camera the *opposite* way - a sign the plugin used to cancel with a second negation elsewhere; both are gone). That is the whole reason this family carries a synthetic scene: the public data cannot show it.

### Misses are NaN, aligned with their input

One last convention worth seeing rather than reading. Point a camera at the horizon and cast the frame border: the top edge misses the terrain. The result keeps its shape and its order; the missing rows are NaN, not silently dropped.

In [ ]:
from bambi.geo.camera import camera_from_pose

# 25 m up, 40 m south of the aim point, tilted 75 degrees: the frame's lower half sees ground,
# its upper half (75 + 25 degrees from nadir) is above the horizon.
oblique = camera_from_pose(scene.aim + [0, -40, 25], [75.0, 0.0, 0.0], scene.fovy)
fp = georef.footprint(oblique, w, h, smesh, samples_per_edge=4)     # 16 border points, clockwise from top-left
print("footprint rows:", fp.shape[0], "  missed:", int(np.isnan(fp).any(axis=1).sum()))
print(np.round(fp, 1))

## Where this goes next

`03_tracking` links detections across frames and geo-references whole tracks with the same two functions; `04_survey_analytics` turns the ground points into densities and population estimates. The parity of all of this with the QGIS plugin's output on a real oblique flight is asserted in `tests/test_parity_georef_plugin.py`.